# Déjame escuchar música

Los datos están almacenados en el archivo `/datasets/music_project_en.csv`.

¡Ahora sí, manos al código!


## Etapa 1. Descripción de los datos <a id='data_review'></a>

In [27]:
import pandas as pd

In [28]:
# Lee el archivo y almacénalo en df
df = pd.read_csv('/datasets/music_project_en.csv')


In [29]:
# Obtén las 10 primeras filas de la tabla df
print(df.head(10))

     userID                        Track            artist   genre  \
0  FFB692EC            Kamigata To Boots  The Mass Missile    rock   
1  55204538  Delayed Because of Accident  Andreas Rönnberg    rock   
2    20EC38            Funiculì funiculà       Mario Lanza     pop   
3  A3DD03C9        Dragons in the Sunset        Fire + Ice    folk   
4  E2DC1FAE                  Soul People        Space Echo   dance   
5  842029A1                       Chains          Obladaet  rusrap   
6  4CB90AA5                         True      Roman Messer   dance   
7  F03E1C1F             Feeling This Way   Polina Griffith   dance   
8  8FA1D3BE                     L’estate       Julia Dalia  ruspop   
9  E772D5C0                    Pessimist               NaN   dance   

        City        time        Day  
0  Shelbyville  20:28:33  Wednesday  
1  Springfield  14:07:09     Friday  
2  Shelbyville  20:58:07  Wednesday  
3  Shelbyville  08:37:09     Monday  
4  Springfield  08:34:34     Monday  
5

In [30]:
# Obtén la información general sobre nuestros datos
print(df.info()) # Línea de código solicitada con  información detallada del DF
print()
print(f'Cantidad de duplicados: {df.duplicated().sum()}') # Aprovechamos para contabilizar duplicados
print()
# Usamos la siguiente línea de código para calcular el porcentaje de duplicados
print(f'Porcentaje de duplicados es: {(df.duplicated().sum()/len(df)*100):.2f}%')
print()
print(df.isna().sum()) # Contabilizamos valores ausentes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0     userID  65079 non-null  object
 1   Track     63736 non-null  object
 2   artist    57512 non-null  object
 3   genre     63881 non-null  object
 4     City    65079 non-null  object
 5   time      65079 non-null  object
 6   Day       65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB
None

Cantidad de duplicados: 3826

Porcentaje de duplicados es: 5.88%

  userID       0
Track       1343
artist      7567
genre       1198
  City         0
time           0
Day            0
dtype: int64


Estas son nuestras observaciones sobre la tabla. Contiene siete columnas que almacenan los mismos tipos de datos: `object`.

Según la documentación:
- `' userID'`: identificador del usuario;
- `'Track'`: título de la canción;
- `'artist'`: nombre del artista;
- `'genre'`: género de la canción;
- `'City'`: ciudad del usuario;
- `'time'`: la hora exacta en la que se reprodujo la canción;
- `'Day'`: día de la semana.

Podemos ver dos problemas con el estilo en los encabezados de la tabla:
1. Algunos encabezados están en mayúsculas, otros en minúsculas.
2. el encabezado ' userID' cuenta con un espacio al principio




### Escribe algunas observaciones por tu parte. Contesta a las siguientes preguntas: <a id='data_review_conclusions'></a>

`1.   ¿Qué tipo de datos hay en las filas? ¿Cómo podemos saber qué almacenan las columnas?`

`2.   ¿Hay suficientes datos para proporcionar respuestas a nuestra hipótesis o necesitamos más información?`

`3.   ¿Notaste algún problema en los datos, como valores ausentes, duplicados o tipos de datos incorrectos?`

Escribe aquí tus respuestas:

1. Los datos almacenados son de tipo object que equivalen a cadenas de texto, también conocidas como strings.  Y, podemos saber qué almacenan las columnas debido a que los encabezados cumplen con la semántica que debe tener el código, puesto que tienen sentido acorde a lo que almacenan, por ejemplo, la columna "Day" muestra el día de la semana o "City" almacena la ciudad del usuario/a 

2. Hay suficente información, puesto que la columna de ciudad incluye valores de las ciudades que se pretenden analizar, así como los días de la semana y la hora específica en la que se reproducen las canciones


3. Hay una cantidad importante de valores nulos en la columna del artista (11,6% del total de filas), esto puede ser un reto si se pretende analizar dicha columna, para el análisis de tiempo y hora, parece no cobrar tanta relevancia.
Por otro lado, existe cerca del 6% de las filas que se podrían clasificar como 'duplicadas'. Estos registros muestra hora y fecha de reproducción de cada canción, por ende, podemos inferir que sí se tratan de duplicados exactos de registros y no de registros de multiples reproducciones en distintas fechas y horas


## Etapa 2. Preprocesamiento de los datos <a id='data_preprocessing'></a>

Tu objetivo aquí es preparar los datos para analizarlos.
El primer paso es resolver los problemas con los encabezados. Después podemos avanzar a los valores ausentes y duplicados. ¡Empecemos!

Vamos a corregir el formato en los encabezados de la tabla.


### Estilo del encabezado <a id='header_style'></a>
Etapa 2.1. Muestra los encabezados de la tabla (los nombres de las columnas):

In [31]:
print(df.columns)


Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='object')


Vamos cambiar los encabezados de la tabla siguiendo las reglas estilísticas convencionales:
*   Todos los caracteres deben ser minúsculas.
*   Elimina los espacios.
*   Si el nombre tiene varias palabras, utiliza snake_case, es decir, añade un guion bajo ( _ ) entre las palabras en lugar de un espacio.



Etapa 2.2. Utiliza el bucle for para iterar sobre los nombres de las columnas y poner todos los caracteres en minúsculas. Cuando hayas terminado, vuelve a mostrar los encabezados de la tabla:

In [32]:
# Bucle que itera sobre los encabezados y los pone todos en minúsculas
new_column_names = [] # creamos lista que almacenará los nuevos nombres
for old_name in df.columns: # iteramos en los nombres de los encabezados
    name_lowered = old_name.lower() # convertimos a minúscula los encabezados
    new_column_names.append(name_lowered) # agregamos los nuevos nombres a la lsita
    
df.columns = new_column_names # Reemplazamos los nombres antiguos

print(df.columns) # Mostramos el resultado


Index(['  userid', 'track', 'artist', 'genre', '  city  ', 'time', 'day'], dtype='object')


Etapa 2.3. Ahora, utilizando el mismo método, elimina los espacios al principio y al final de los nombres de las columnas y muestra los nombres de las columnas de nuevo:

In [33]:
# Bucle que itera sobre los encabezados y elimina los espacios
new_column_names = [] # creamos lista que almacenará los nuevos nombres
for old_name in df.columns: # iteramos en los nombres de los encabezados
    name_lowered = old_name.lower() # convertimos a minúscula los encabezados
    name_stripped = name_lowered.strip() # Eliminamos los espacios antes y después de cada header
    new_column_names.append(name_stripped) # agregamos los nuevos nombres a la lsita
    
df.columns = new_column_names # Reemplazamos los nombres antiguos

print(df.columns) # Mostramos el resultado


Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


Etapa 2.4. Necesitamos aplicar la regla de snake_case en la columna `userid`. Debe ser `user_id`. Cambia el nombre de esta columna y muestra los nombres de todas las columnas cuando hayas terminado.

In [34]:
# Cambia el nombre de la columna "userid"
# usamos el método #rename()" y el atributo "inplace" para reemplazar el nombre
# directamente en los nombres de las columnas
df.rename(columns={'userid': 'user_id'}, inplace=True)
print(df.columns)
#Se mantiene uso de "rename()", pese al comentatio del bot, considerando que la pista solicita explícitamente usar este método

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


Etapa 2.5. Comprueba el resultado. Muestra los encabezados una vez más:

In [35]:
# Comprueba el resultado: lista de encabezados
print(df.columns)

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


### Valores ausentes <a id='missing_values'></a>
 Etapa 2.5. Primero, encuentra el número de valores ausentes en la tabla. Debes utilizar dos métodos para obtener el número de valores ausentes.

In [36]:
# Calcula el número de valores ausentes
print(f'valores ausentes:\n {df.isna().sum()}')
"""
Usamos el método isna() para extraer los valores ausentes
y el método sum() para contabilizarlos
"""
# El bot indica usar un segundo método para contar valores ausentes,
#sin embaego, interpreto que el enunciado, al decir "debes utilizar dos métodos"
#se refiere al uso conjunto de isna y sum. Aún así, uso el método "info()" 
# como segunda opción para mostrar el resultado requerido
print(df.info())


valores ausentes:
 user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  65079 non-null  object
 1   track    63736 non-null  object
 2   artist   57512 non-null  object
 3   genre    63881 non-null  object
 4   city     65079 non-null  object
 5   time     65079 non-null  object
 6   day      65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB
None


Etapa 2.6. Sustituye los valores ausentes en las columnas `'track'`, `'artist'` y `'genre'` con el string `'unknown'`.

1. Crea una lista llamada columns_to_replace que contenga los nombres de las columnas 'track', 'artist' y 'genre'.

2. Usa un bucle for para iterar sobre cada columna en columns_to_replace.

3. Dentro del bucle, sustituye los valores ausentes en cada columna con el string `'unknown'`.

In [37]:
# Bucle en los encabezados reemplazando los valores ausentes con 'unknown'
columns_to_replace = ['track', 'artist', 'genre'] # Creamos lista con columnas a iterar
for column in columns_to_replace: # creamos el iterador
    df[column].fillna('unknown',inplace=True) # pedimos que reemplace los valores ausentes con el string indicado
print(df.isna().sum())

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


Etapa 2.7. Ahora comprueba el resultado para asegurarte de que no falten valores ausentes por reemplazar en el conjunto de datos. Para ello, cuenta los valores ausentes una vez más.

In [38]:
# Cuenta los valores ausentes

print(df.isna().sum())


user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


### Duplicados <a id='duplicates'></a>
Etapa 2.8. Encuentra el número de duplicados explícitos en la tabla. Una vez más, debes aplicar dos métodos para obtener la cantidad de duplicados explícitos.

In [39]:
# Cuenta los duplicados explícitos
print(f'Cantidad de duplicados (método 1): {df.duplicated().sum()}')
# mismo caso que el de arriba con el bot, se investiga y se utiliza máscara booleana para segundo conteo como alternativa adicional
df_extension = len(df)
df_extension_without_duplicates = len(df.drop_duplicates())
final_result = df_extension - df_extension_without_duplicates
print(f'Cantidad de duplicados (método 2): {final_result}')


Cantidad de duplicados (método 1): 3826
Cantidad de duplicados (método 2): 3826


Etapa 2.9. Ahora, elimina todos los duplicados. Para ello, llama al método que hace exactamente esto.

In [40]:
# Elimina los duplicados explícitos
df = df.drop_duplicates().reset_index(drop=True) # eliminamos duplicados con drop_duplicates y restauramos los índices, almacenamos el resultado en el DataFrame nuevamente
print(df.duplicated().sum())

0


Etapa 2.10. Comprobemos ahora si conseguimos eliminar todos los duplicados. Cuenta los duplicados explícitos una vez más para asegurarte de haberlos eliminado todos:

In [41]:
# Comprueba de nuevo si hay duplicados
print(df.duplicated().sum())

0


Ahora queremos deshacernos de los duplicados implícitos en la columna `genre`. Por ejemplo, el nombre de un género se puede escribir de varias formas. Dichos errores también pueden afectar al resultado.

Etapa 2.11. Primero debemos mostrar una lista de nombres de géneros únicos, por orden alfabético. Para ello:
1. Extrae la columna `genre` del DataFrame.
2. Llama al método que devolverá todos los valores únicos en la columna extraída.


In [42]:
# Inspecciona los nombres de géneros únicos
print(sorted(df['genre'].unique())) # Usamos unique para extraer los valores únicos de la columna señalada

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

Etapa 2.12. Vamos a examinar la lista para identificar **duplicados implícitos** del género `hiphop`, es decir, nombres mal escritos o variantes que hacen referencia al mismo género musical.

Los duplicados que encontrarás son:

* `hip`  
* `hop`  
* `hip-hop`  

Para solucionarlo, vamos a crear una función llamada `replace_wrong_values()`.


1. Define una función llamada `replace_wrong_values()` que reciba los siguientes parámetros:

* `df`: el DataFrame a modificar
* `column`: el nombre de la columna a trabajar
* `wrong_values`: una lista con los valores incorrectos
* `correct_value`: el valor correcto para reemplazar

2. Dentro de la función, usa un bucle `for` para iterar sobre cada valor incorrecto y aplicar `.replace()`.


In [43]:

# Función para reemplazar los duplicados implícitos
def replace_wrong_values(df, column, wrong_values, correct_value): # Creamos función que recibe los parámetros requeridos
    for wrong_value in wrong_values: # Comenzamos a iterar sobre los valores errados
        df[column] = df[column].replace(wrong_value, correct_value) # Reemplazamos los valores
    return df # Devolvemos el dataframe ya procesado 


Etapa 2.13. Ahora, llama a la función pasando:

* `df` como el DataFrame
* `'genre'` como nombre de columna
* `['hip', 'hop', 'hip-hop']` como lista de valores incorrectos
* `'hiphop'` como valor correcto


In [44]:
# Elimina los duplicados implícitos
replace_wrong_values(df,'genre',['hip', 'hop', 'hip-hop'],'hiphop') # llamamos función y proveems los parámetros requeridos

,user_id,track,artist,genre,city,time,day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
...,...,...,...,...,...,...,...
61248,729CBB09,My Name,McLean,rnb,Springfield,13:32:28,Wednesday
61249,D08D4A55,Maybe One Day (feat. Black Spade),Blu & Exile,hiphop,Shelbyville,10:00:00,Monday
61250,C5E3A0D5,Jalopiina,unknown,industrial,Springfield,20:09:26,Friday
61251,321D0506,Freight Train,Chas McDevitt,rock,Springfield,21:43:59,Friday


Etapa 2.14. Asegúrate de que los nombres duplicados se hayan eliminado. Muestra la lista de valores únicos de la columna `'genre'` una vez más:

In [45]:
# Comprueba de nuevo los duplicados implícitos
print(sorted(df['genre'].unique())) # Imprimimos listado de valores únicos ordenados de manera alfabética

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

## Etapa 3. Análisis

### Tarea: Comparar el comportamiento de los usuarios en las dos ciudades <a id='activity'></a>

Queremos analizar si hay diferencias en la cantidad de canciones reproducidas en Springfield y Shelbyville. Para ello, usaremos los datos de dos días de la semana: lunes y viernes.

Compararemos cuántas canciones se escucharon en cada ciudad durante esos días para identificar posibles patrones de comportamiento.

Sigue estos tres pasos para organizar tu análisis:

- Dividir: agrupa los datos por ciudad.

- Aplicar: cuenta cuántas canciones se reproducen en cada grupo.

- Combinar: presenta los resultados de forma que se puedan comparar fácilmente ambas ciudades.

Repite este proceso por separado para cada uno de los dos días.

Etapa 3.1.
Cuenta cuántas canciones se reprodujeron en cada ciudad utilizando la columna `'track'` como referencia.

In [46]:
# Cuenta las canciones reproducidas en cada ciudad
print(df.groupby('city')['track'].count())
print(df.columns)

city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64
Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


Etapa 3.2. Redacta brevemente tus observaciones sobre los resultados.

¿Qué diferencias encontraste entre Springfield y Shelbyville? ¿A qué podrían deberse?

Escribe tus observaciones aquí.

Springfield tiene la mayor participación con 69,7% del total de registros entre las 2 ciudades. Esto podría estar relacionado al tamaño de la población, si Spriengfield tiene un tamaño mayor, esto pudiera sugerir una relación directamente proporcional. También se podría hacer un análisis de edades y género escuchados para deducir si, de acuerdo a la distribuciuón generacional, puede haber una mayor adopción de plataformas de streaming en un lugar que en el otro. Inicialmente mi hipótesis es el tamaño de la población y secundaria, una penetración en el mercado más amplia en springfield de acuerdo al factor "age" (Se necesita más data para comprobar esta hipótesis)

Etapa 3.3.
Agrupa los datos por día de la semana y cuenta cuántas canciones se reprodujeron los lunes y viernes.



In [47]:
# Calcula las canciones reproducidas en cada uno de los dos días
days_to_search = df[df['day'].isin(['Monday', 'Friday'])] # primero filtramos los 2 días que queremos extraer
songs_per_day = days_to_search.groupby('day')['track'].count() # agrupamos por día la variable que acabamos de crear para extraer data de solo lunes y viernes
print(songs_per_day) # Imprimimos resultado final

day
Friday    21840
Monday    21354
Name: track, dtype: int64


Etapa 3.4. Describe brevemente qué observaste al comparar los lunes y viernes.

¿Hubo un día con más actividad? ¿Cambia algo si analizas cada ciudad por separado?

Escribe tus observaciones aquí.
Ambos días son mjuy similares en cuanto a reproducciones, siendo el viernes ligeramente mayor por cerca de 500 reproducciones registradas. A nivel de ciudad, existe una diferencia significativa entre ambas, siendo Springfield aquella que tiene una participación mayoritaria

Etapa 3.5

Ahora vamos a combinar dos criterios: día y ciudad.

Crea una función llamada `number_tracks()` que reciba dos parámetros:

* `day`: un día de la semana (por ejemplo, `'Monday'`)
* `city`: el nombre de una ciudad (por ejemplo, `'Springfield'`)

Dentro de la función:

1. Filtra el DataFrame por el día.
2. Luego, filtra por la ciudad.
3. Cuenta cuántas veces aparece `'user_id'` en ese filtro.
4. Devuelve ese número como resultado.


In [48]:
# Declara la función number_tracks() con dos parámetros: day= y city=.


def number_tracks(day, city):
    
    # Almacena las filas del DataFrame donde el valor en la columna 'day' es igual al parámetro day=
    day_rows = df[df['day'] == day]
    # Filtra las filas donde el valor en la columna 'city' es igual al parámetro city=
    columns_city = day_rows[day_rows['city'] == city]
    # Extrae la columna 'user_id' de la tabla filtrada y aplica el método count()
    final_count = columns_city['user_id'].count()
    # Devuelve el número de valores de la columna 'user_id'
    return final_count


Etapa 3.6. Llama a `number_tracks()` cuatro veces: una por ciudad en cada uno de los dos días.

In [49]:
# El número de canciones reproducidas en Springfield el lunes

springfield_monday = number_tracks('Monday','Springfield')
print(springfield_monday)



15740


In [50]:
# El número de canciones reproducidas en Shelbyville el lunes

shelbyville_monday = number_tracks('Monday','Shelbyville')
print(shelbyville_monday)




5614


In [51]:
# El número de canciones reproducidas en Springfield el viernes
springfield_friday = number_tracks('Friday','Springfield')
print(springfield_friday)

15945


In [52]:
# El número de canciones reproducidas en Shelbyville el viernes
shelbyville_friday = number_tracks('Friday','Shelbyville')
print(shelbyville_friday)

5895


# Conclusiones <a id='end'></a>

## Escribe tus conclusiones finales sobre el análisis

Redacta un resumen breve y claro de los hallazgos obtenidos durante el proceso de análisis. Tu conclusión debe:

* Mencionar los principales patrones que observaste en los datos.
* Identificar cualquier problema encontrado y cómo lo solucionaste.
* Explicar cómo estas acciones ayudaron a mejorar la calidad del análisis.

Reflexiona también sobre la pregunta central:

> *¿Los datos muestran que el comportamiento de los usuarios —en cuanto a la música que escuchan— varía según la ciudad y el día de la semana?*

Apoya tu respuesta con ejemplos concretos de los resultados obtenidos.


# Resumen ejecutivo
## Objetivo del análisis
- Analizar tendencias de reproducción de música en una plataforma de streaming para obtener información detallada de los principales comportamientos de la población
## Principales hallazgos
- Springfield es la ciudad que mayor contribución en términos de consumo tiene para la plataforma de música en straming
- No existe variación significativa entre días de la semana, los lunes y viernes el total de reproducciones es superior a 21k
## Recomendaciones principales
- se puede incliuir una columna con el atributo "age" para ampliar el análisis y determinar el nivel de influencia que esto puede llegar a tener en el comportamiento de la población por ciudad y día
- Se puede indagar más a profundidad en los artistas más escuchados por día y por intervalo específico del día para entender los comportamientos de consumo de los usuarios, así comprobar hipótesis como que los viernes en horas nocturnas puede haber un decremento debido a actividades nocturnas, especialmente si, al agregar el factor "age" notamos una participación significativa de personas con edad entre 18 y 25 años, los cuáles sueles participar en actividades sociales que no necesariemente promueven el consumo individual de música en streaming, por lo tanto, canalizar los esfuerzos de marketing en otra población para aumentar los patrones de consumo en dichas franjas
# Descripción del Dataset
## Fuente de datos
- Dataset de un total de 65078 registros de 2 ciudades /Springfield y Shelbyville para 3 dias de la semana (Monday, wednesday & friday) con datos de tipo 'object' distribuidas en las siguientes columnas ['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day']
# Variaciones principales analizadas
- Se analizan los registros de consumo para tratar de encontrar diferenciass significativas a nivel de día de la semana y ciudad
# Calidad de los datos
## valores faltantes identificados
- Se identifican un total de 10108 valores ausentes distribuidos en 3 columnas ('track', 'artis', 'genre') siendo 'artist' la columna con más valores ausentes con 7567 que representan más del 70% de valores ausentes
## Duplicados encontrados y tratados
- También se identifican 3826 duplicados, que son poco más del 5% del total de registros en el Dataset
## Acciones de limpieza realizadas
- Se aplican métodos al DataFrame como dupplicated().sum() y drop_duplicates() para contabilizar y eliminar duoplicados, así como isna().sum() para contabilizar valores ausentes y luego diligenciarlos con el método fillna(), a su vez, restauramos los índices con el método reset_index()
# Análisis descriptivo
## Por ciudad
### Actividad total de reproducciones
- Springfield tiene una participación 1 veces superior a Shelbiville en cuando a cantidad de reproducciones, lo cual puede estar relacionado con el tamaño poblacional o los hábitos de consumo según la distribución generacional de cada ciudad
## Por día de la semana
### Días con mayor/menos actividad
- Los días lunes y viernes tienen un comportamiento muy similar en cuanto a total de reproducciones, al tener una diferencia menor a 500 en el total de registros y elevándose por encima de las 21k reproducciones por día.
- Los días de menor actividad en general son los miércoles, donde el número deciende cerca a los 18k
## Comportamiento de los usuarios

*¿Los datos muestran que el comportamiento de los usuarios —en cuanto a la música que escuchan— varía según la ciudad y el día de la semana?*

Sí lo muestran aunque de una manera superficial y con leves diferencias, lo que da espacio a especulaciones que no pueden ser comprobadas al 100% debido a la falta de información más detallada. No obstante, con el Dataset suministrado, podemos encontar patrones a nivel de hora y día y ciudades específicas en las cuales, si bien a nivel general se evidencia una tendencia muy estática, sí existe un gap enorme entre Springfield (mayor participación) y Shelbyville.

